In [ ]:
import networkx as nx
import numpy as np


# ------------------------------------------------------------
# STEP 1: Load Zachary's Karate Club dataset
# ------------------------------------------------------------
# NetworkX provides Zachary's Karate Club network.
#
# Each node represents a member of the karate club.
# An edge represents a friendship/interaction between two
# members.
# ------------------------------------------------------------

G = nx.karate_club_graph()

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

Number of nodes: 34
Number of edges: 78


In [ ]:

# ------------------------------------------------------------
# STEP 2: Create the adjacency matrix A
# ------------------------------------------------------------
# A[i][j] = 1 if there is an edge between node i and node j.
# A[i][j] = 0 otherwise.
#
# We first create a fixed list of nodes so that the rows and
# columns of all matrices correspond to the same nodes.
# ------------------------------------------------------------

nodes = list(G.nodes())
n = len(nodes)

# Create an n x n matrix filled with zeros
A = np.zeros((n, n))

# Fill the adjacency matrix
for i in range(n):
    for j in range(n):

        if G.has_edge(nodes[i], nodes[j]):
            A[i][j] = 1


print("\nAdjacency Matrix A:")
print(A)


# ------------------------------------------------------------
# STEP 3: Find the neighbourhood of every node
# ------------------------------------------------------------
# The neighbourhood N(i) is the set of nodes directly connected
# to node i.
#
# Example:
# If node 1 is connected to nodes 2, 3 and 4:
#
# N(1) = {2, 3, 4}
# ------------------------------------------------------------

neighborhoods = {}

for node in nodes:
    neighborhoods[node] = set(G.neighbors(node))


Adjacency Matrix A:
[[0. 1. 1. ... 1. 0. 0.]
 [1. 0. 1. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 1. 0.]
 ...
 [1. 0. 0. ... 0. 1. 1.]
 [0. 0. 1. ... 1. 0. 1.]
 [0. 0. 0. ... 1. 1. 0.]]


In [ ]:
# ------------------------------------------------------------
# STEP 4: Calculate the Jaccard Dissimilarity Matrix D
# ------------------------------------------------------------
#
# The formula used in the presentation is:
#
#              |N(i) ∩ N(j)|
# D(i,j) = 1 - ----------------
#              |N(i) ∪ N(j)|
#
# Intersection = neighbours shared by i and j
# Union        = all neighbours belonging to i or j
#
# D(i,j) = 0  -> identical neighbourhoods
# D(i,j) = 1  -> no common neighbours
#
# We calculate D only for existing edges because W will only
# assign weights to existing connections.
# ------------------------------------------------------------

D = np.zeros((n, n))

for i in range(n):
    for j in range(n):

        # Calculate dissimilarity only when an edge exists
        if A[i][j] == 1:

            # Get neighbourhoods of nodes i and j
            Ni = neighborhoods[nodes[i]]
            Nj = neighborhoods[nodes[j]]

            # Find common neighbours
            intersection = Ni.intersection(Nj)

            # Find all unique neighbours
            union = Ni.union(Nj)

            # Calculate Jaccard dissimilarity
            if len(union) == 0:
                D[i][j] = 0
            else:
                D[i][j] = 1 - (len(intersection) / len(union))


print("\nDissimilarity Matrix D:")
print(D)



Dissimilarity Matrix D:
[[0.         0.61111111 0.76190476 ... 1.         0.         0.        ]
 [0.61111111 0.         0.73333333 ... 0.         0.         0.        ]
 [0.76190476 0.73333333 0.         ... 0.         0.95238095 0.        ]
 ...
 [1.         0.         0.         ... 0.         0.94117647 0.9047619 ]
 [0.         0.         0.95238095 ... 0.94117647 0.         0.47368421]
 [0.         0.         0.         ... 0.9047619  0.47368421 0.        ]]


In [ ]:


# ------------------------------------------------------------
# STEP 5: Create the weighted matrix W
# ------------------------------------------------------------
#
# According to the presentation:
#
#       W = A o D
#
# where "o" represents element-wise multiplication.
#
# Therefore:
#
#       W(i,j) = A(i,j) * D(i,j)
#
# If there is no edge:
#       A(i,j) = 0 -> W(i,j) = 0
#
# If an edge exists:
#       W(i,j) = D(i,j)
#
# Thus, every edge receives a weight based on the structural
# dissimilarity of its endpoints.
# ------------------------------------------------------------

W = A * D


print("\nDissimilarity-Weighted Matrix W:")
print(W)



Dissimilarity-Weighted Matrix W:
[[0.         0.61111111 0.76190476 ... 1.         0.         0.        ]
 [0.61111111 0.         0.73333333 ... 0.         0.         0.        ]
 [0.76190476 0.73333333 0.         ... 0.         0.95238095 0.        ]
 ...
 [1.         0.         0.         ... 0.         0.94117647 0.9047619 ]
 [0.         0.         0.95238095 ... 0.94117647 0.         0.47368421]
 [0.         0.         0.         ... 0.9047619  0.47368421 0.        ]]


In [ ]:


# ------------------------------------------------------------
# STEP 6: Find the eigenvalues and eigenvectors of W
# ------------------------------------------------------------
#
# The presentation gives:
#
#       Wc = lambda_max c
#
# We need the eigenvector corresponding to the largest
# eigenvalue.
#
# This eigenvector gives the dissimilarity-based centrality
# scores.
# ------------------------------------------------------------

eigenvalues, eigenvectors = np.linalg.eig(W)


# ------------------------------------------------------------
# STEP 7: Find the dominant eigenvalue
# ------------------------------------------------------------
# np.argmax() gives the position of the largest eigenvalue.
# ------------------------------------------------------------

largest_index = np.argmax(eigenvalues)

largest_eigenvalue = eigenvalues[largest_index]

print("\nLargest Eigenvalue:")
print(largest_eigenvalue)



Largest Eigenvalue:
(5.733269044986493+0j)


In [ ]:


# ------------------------------------------------------------
# STEP 8: Extract the corresponding eigenvector
# ------------------------------------------------------------

centrality = np.real(eigenvectors[:, largest_index])


# ------------------------------------------------------------
# STEP 9: Make all centrality values positive
# ------------------------------------------------------------
# Eigenvectors can sometimes be returned with negative values.
# Multiplying the complete eigenvector by -1 does not change
# its meaning.
#
# We make the values positive for easier interpretation.
# ------------------------------------------------------------

if np.sum(centrality) < 0:
    centrality = -centrality


# ------------------------------------------------------------
# STEP 10: Normalize the centrality scores
# ------------------------------------------------------------
# We divide by the largest score so that the most central node
# has a score of 1.
#
# This makes the results easier to compare.
# ------------------------------------------------------------

centrality = centrality / np.max(centrality)


# ------------------------------------------------------------
# STEP 11: Create a list containing:
#
# Node number
# Centrality score
#
# Then sort the nodes from highest centrality to lowest.
# ------------------------------------------------------------

results = []

for i in range(n):
    results.append((nodes[i], centrality[i]))


results.sort(key=lambda x: x[1], reverse=True)


# ------------------------------------------------------------
# STEP 12: Display the final ranking
# ------------------------------------------------------------


print("DISSIMILARITY-BASED CENTRALITY RANKING")
print("Zachary's Karate Club Network")


print("\nRank\tNode\tCentrality")
print("--------------------------------")

rank = 1

for node, score in results:

    print(rank, "\t", node, "\t", round(score, 6))

    rank += 1

DISSIMILARITY-BASED CENTRALITY RANKING
Zachary's Karate Club Network

Rank	Node	Centrality
--------------------------------
1 	 33 	 1.0
2 	 0 	 0.765913
3 	 32 	 0.763331
4 	 2 	 0.693055
5 	 8 	 0.543817
6 	 31 	 0.52476
7 	 1 	 0.502948
8 	 13 	 0.477086
9 	 30 	 0.425658
10 	 23 	 0.391756
11 	 19 	 0.380339
12 	 27 	 0.375967
13 	 28 	 0.366211
14 	 3 	 0.338288
15 	 29 	 0.337842
16 	 9 	 0.295304
17 	 7 	 0.293055
18 	 14 	 0.28763
19 	 15 	 0.28763
20 	 18 	 0.28763
21 	 20 	 0.28763
22 	 22 	 0.28763
23 	 26 	 0.211872
24 	 17 	 0.204685
25 	 21 	 0.204685
26 	 12 	 0.176308
27 	 25 	 0.172094
28 	 24 	 0.169678
29 	 5 	 0.16905
30 	 6 	 0.16905
31 	 4 	 0.165546
32 	 10 	 0.165546
33 	 11 	 0.133591
34 	 16 	 0.047177
